# 06 · RAG Evaluation

Is the RAG pipeline actually doing anything useful? Compare it against a 'vanilla' LLM (same model, no retrieval) on a small golden-set of plant-disease treatment questions.

**Metric**: Is the answer grounded in the specific facts present in our knowledge base? We score by checking whether key phrases from the KB entry appear in the answer.

This setup mirrors the example project's evaluation approach.

## Setup

In [ ]:
import sys, os
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv('../.env')

import json, pandas as pd
from langchain_openai import ChatOpenAI
from src.rag import PlantDocRAG

rag = PlantDocRAG(persist_directory='../models/chroma_db')
vanilla_llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.2)

## Golden set

A mix of questions: symptoms, treatment specifics, prevention strategies. Each has a list of 'key phrases' that a correct grounded answer should contain.

In [ ]:
golden_set = [
    {'q': 'What fungicide should I use for apple scab?',
     'class_filter': 'Apple___Apple_scab',
     'key_phrases': ['captan', 'myclobutanil', 'sulfur']},
    {'q': 'Is there a cure for citrus greening?',
     'class_filter': 'Orange___Haunglongbing_(Citrus_greening)',
     'key_phrases': ['no cure', 'psyllid', 'remove']},
    {'q': 'How do I prevent potato late blight?',
     'class_filter': 'Potato___Late_blight',
     'key_phrases': ['certified', 'seed', 'resistant']},
    {'q': 'What does early blight look like on tomato?',
     'class_filter': 'Tomato___Early_blight',
     'key_phrases': ['concentric', 'ring', 'older leaves']},
    {'q': 'Which virus causes curled yellow leaves in tomato?',
     'class_filter': 'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
     'key_phrases': ['whitefly', 'TYLCV', 'resistant']},
    {'q': 'Best treatment for grape black rot?',
     'class_filter': 'Grape___Black_rot',
     'key_phrases': ['mancozeb', 'mummified', 'fungicide']},
    {'q': 'How to manage spider mites on tomato?',
     'class_filter': 'Tomato___Spider_mites Two-spotted_spider_mite',
     'key_phrases': ['insecticidal soap', 'predatory', 'water']},
    {'q': 'What resistant varieties exist for apple scab?',
     'class_filter': 'Apple___Apple_scab',
     'key_phrases': ['Liberty', 'Enterprise', 'Freedom']},
    {'q': 'How do I control powdery mildew on squash organically?',
     'class_filter': 'Squash___Powdery_mildew',
     'key_phrases': ['potassium bicarbonate', 'neem', 'milk']},
    {'q': 'What temperatures favor corn gray leaf spot?',
     'class_filter': 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
     'key_phrases': ['75', '85', 'humidity']},
]
print(f'Golden set size: {len(golden_set)}')

## Score function

In [ ]:
def grounding_score(answer: str, key_phrases: list) -> float:
    '''Fraction of key phrases that appear (case-insensitive) in the answer.'''
    if not key_phrases: return 1.0
    answer_lower = answer.lower()
    hits = sum(1 for kp in key_phrases if kp.lower() in answer_lower)
    return hits / len(key_phrases)

## Evaluation loop

In [ ]:
rows = []
for item in golden_set:
    # RAG answer
    rag_res = rag.answer(item['q'], class_filter=item['class_filter'])
    rag_answer = rag_res['answer']
    rag_score = grounding_score(rag_answer, item['key_phrases'])

    # Vanilla LLM answer (no retrieval)
    vanilla_answer = vanilla_llm.invoke(item['q']).content
    vanilla_score = grounding_score(vanilla_answer, item['key_phrases'])

    rows.append({
        'question': item['q'],
        'rag_score': rag_score,
        'vanilla_score': vanilla_score,
        'rag_answer': rag_answer[:200],
        'vanilla_answer': vanilla_answer[:200],
    })

df = pd.DataFrame(rows)
df

## Aggregate results

In [ ]:
print(f'Mean RAG grounding score:     {df["rag_score"].mean()*100:.1f}%')
print(f'Mean Vanilla grounding score: {df["vanilla_score"].mean()*100:.1f}%')
print(f'Delta (RAG - Vanilla):        {(df["rag_score"].mean() - df["vanilla_score"].mean())*100:.1f} pp')
df.to_csv('../docs/rag_evaluation_results.csv', index=False)

## Takeaways

Expected: RAG significantly outperforms vanilla because the KB contains specific facts (fungicide names, variety names, exact temperature ranges) that a general LLM won't reliably reproduce. Vanilla may give generally correct but vague answers that miss the specific phrases.

Fill in actual numbers after running.